In [22]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.preprocessing import StandardScaler
from scipy.interpolate import PchipInterpolator
import warnings
warnings.filterwarnings('ignore')

np.random.seed(42)
torch.manual_seed(42)

In [23]:
RAW_CSV    = '../data/raw/dataset.csv'
OUTPUT_CSV = '../outputs/5_submissions/filled_dataset.csv'
SUBMIT_CSV = '../outputs/5_submissions/submission.csv'

SM    = 0.60
EW    = 0.70
ALPHA = 0.05   # weight of PCHIP vs cross-sectional in expiry blend

# MLP training params
EPOCHS       = 500
BATCH_SIZE   = 256
PATIENCE     = 40
LR           = 5e-4
WEIGHT_DECAY = 1e-5

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

FEATURE_COLS = [
    'iv_roll_mean_10', 'iv_roll_mean_5',
    'iv_neighbor_mean', 'wide_iv_neighbor_mean',
    'strike',
    'iv_neighbor_+1', 'iv_neighbor_-1',
    'iv_neighbor_+2', 'iv_neighbor_-2',
    'dist_from_atm_pct', 'moneyness', 'log_moneyness',
    'is_ce',
]

In [24]:
# 1. LOAD RAW DATA
def load_raw():
    df = pd.read_csv(RAW_CSV)
    df['datetime'] = pd.to_datetime(df['datetime'], format='%d-%m-%Y %H:%M')
    df = df.sort_values('datetime').reset_index(drop=True)
    return df


def get_strike(col):
    return int(col.replace('NIFTY27JAN26', '')[:-2])


def get_option_cols(df):
    ce_cols = sorted([c for c in df.columns if c.endswith('CE')], key=get_strike)
    pe_cols = sorted([c for c in df.columns if c.endswith('PE')], key=get_strike)
    return ce_cols, pe_cols

In [25]:
# 2. EXPIRY DAY FILL  (cross-sectional linear + PCHIP blend)
def fill_wide(df, ce_cols, pe_cols):
    """
    Cross-sectional linear interpolation + PCHIP temporal blend
    for expiry day rows only.
    """
    # fix: always work on a clean 0-based RangeIndex so .loc[ti, ...] is safe
    df = df.reset_index(drop=True)

    iv_cols    = ce_cols + pe_cols
    ce_strikes = np.array([get_strike(c) for c in ce_cols])
    pe_strikes = np.array([get_strike(c) for c in pe_cols])
    spot_vals  = df['underlying_price'].values
    N          = len(df)

    dates        = df['datetime'].dt.date
    all_dates    = sorted(dates.unique())
    expiry_date  = all_dates[-1] # last day only
    expiry_mask  = (dates == expiry_date).values

    # Pass 1: cross-sectional fill (expiry rows only)
    filled_cross = df[iv_cols].copy()

    for ti in range(N):
        if not expiry_mask[ti]:
            continue

        spot = spot_vals[ti]

        for cols_grp, strikes_arr in [(ce_cols, ce_strikes), (pe_cols, pe_strikes)]:
            row_vals = filled_cross.loc[ti, cols_grp].values.astype(float)
            obs  = ~np.isnan(row_vals)
            miss = np.isnan(row_vals)
            if obs.sum() < 4 or miss.sum() == 0:
                continue

            moneyness = np.log(strikes_arr / spot)
            obs_x  = moneyness[obs]
            obs_y  = row_vals[obs]
            sort_i = np.argsort(obs_x)
            obs_xs = obs_x[sort_i]
            obs_ys = obs_y[sort_i]

            for mi in np.where(miss)[0]:
                tgt_m      = moneyness[mi]
                left_mask  = obs_xs < tgt_m
                right_mask = obs_xs > tgt_m
                has_left   = left_mask.any()
                has_right  = right_mask.any()

                if has_left and has_right:
                    lx, ly = obs_xs[left_mask][-1], obs_ys[left_mask][-1]
                    rx, ry = obs_xs[right_mask][0],  obs_ys[right_mask][0]
                    t      = (tgt_m - lx) / (rx - lx) if abs(rx - lx) > 1e-10 else 0.5
                    pred   = ly + t * (ry - ly)

                elif has_left:
                    lx2, ly2 = obs_xs[left_mask][-1], obs_ys[left_mask][-1]
                    if left_mask.sum() >= 2:
                        lx1, ly1 = obs_xs[left_mask][-2], obs_ys[left_mask][-2]
                        slope    = (ly2 - ly1) / (lx2 - lx1 + 1e-10)
                        slope    = min(slope, abs(ly2 / (lx2 + 1e-10)) * 1.5)
                    else:
                        slope = 0
                    pred = ly2 + slope * (tgt_m - lx2)

                else:
                    rx2, ry2 = obs_xs[right_mask][0], obs_ys[right_mask][0]
                    if right_mask.sum() >= 2:
                        rx1, ry1 = obs_xs[right_mask][1], obs_ys[right_mask][1]
                        slope    = (ry1 - ry2) / (rx1 - rx2 + 1e-10)
                        slope    = min(slope, abs(ry2 / (abs(rx2) + 1e-10)) * 1.5)
                    else:
                        slope = 0
                    pred = ry2 + slope * (tgt_m - rx2)

                filled_cross.loc[ti, cols_grp[mi]] = max(pred, 0.005)

    filled_cross = filled_cross.ffill().bfill()

    # Pass 2: PCHIP temporal fill (expiry rows only)
    filled_temp    = df[iv_cols].copy()
    expiry_indices = np.where(expiry_mask)[0]

    for col in iv_cols:
        series        = df[col].copy()
        series_expiry = series.iloc[expiry_indices]
        obs_mask      = ~series_expiry.isna()
        ox            = expiry_indices[obs_mask.values]
        oy            = series_expiry[obs_mask].values
        miss_x        = expiry_indices[~obs_mask.values]
        if len(miss_x) == 0 or len(ox) < 2:
            continue
        pi = PchipInterpolator(ox, oy, extrapolate=True)
        filled_temp.loc[miss_x, col] = np.clip(pi(miss_x), 0.005, None)

    filled_temp = filled_temp.ffill().bfill()

    # Blend (expiry rows only)\
    final     = df[iv_cols].copy()
    miss_orig = df[iv_cols].isna()

    for col in iv_cols:
        miss = miss_orig[col].values & expiry_mask
        final.loc[miss, col] = (
            (1 - ALPHA) * filled_cross.loc[miss, col].values
            + ALPHA     * filled_temp.loc[miss, col].values
        )

    final = final.clip(lower=0.005)
    return final, expiry_mask


fill_wide_df = fill_wide

In [26]:
# 3. MLP
class IVSurfaceMLP(nn.Module):
    def __init__(self, input_dim, hidden_dims=None, dropout=0.1):
        super().__init__()
        if hidden_dims is None:
            hidden_dims = [128, 64, 32]
        layers = []
        prev   = input_dim
        for h in hidden_dims:
            layers += [nn.Linear(prev, h), nn.BatchNorm1d(h), nn.GELU(), nn.Dropout(dropout)]
            prev = h
        layers.append(nn.Linear(prev, 1))
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        return self.net(x).squeeze(-1)


def train_mlp(X_train, y_train, X_val, y_val,
              epochs=EPOCHS, batch_size=BATCH_SIZE, patience=PATIENCE):
    scaler      = StandardScaler()
    X_train_trf = scaler.fit_transform(X_train)
    X_val_trf   = scaler.transform(X_val)

    Xtt = torch.tensor(X_train_trf, dtype=torch.float32).to(DEVICE)
    ytt = torch.tensor(y_train,     dtype=torch.float32).to(DEVICE)
    Xvt = torch.tensor(X_val_trf,   dtype=torch.float32).to(DEVICE)
    yvt = torch.tensor(y_val,       dtype=torch.float32).to(DEVICE)

    loader  = DataLoader(TensorDataset(Xtt, ytt), batch_size=batch_size, shuffle=True)
    model   = IVSurfaceMLP(X_train.shape[1]).to(DEVICE)
    opt     = torch.optim.Adam(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    sched   = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs)
    loss_fn = nn.MSELoss()

    best_loss, best_state, no_imp = float('inf'), None, 0
    best_epoch = 0
    train_losses, val_losses = [], []

    for epoch in range(epochs):
        model.train()
        epoch_loss = 0
        for xb, yb in loader:
            opt.zero_grad()
            loss = loss_fn(model(xb), yb)
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()
            epoch_loss += loss.item()
        sched.step()

        model.eval()
        with torch.no_grad():
            vl = loss_fn(model(Xvt), yvt).item()

        train_losses.append(epoch_loss / len(loader))
        val_losses.append(vl)

        if vl < best_loss:
            best_loss  = vl
            best_epoch = epoch
            best_state = {k: v.clone() for k, v in model.state_dict().items()}
            no_imp     = 0
        else:
            no_imp += 1
            if no_imp >= patience:
                break

    model.load_state_dict(best_state)
    return model, scaler, best_epoch, np.array(train_losses), np.array(val_losses)


def predict_mlp(model, scaler, X):
    model.eval()
    with torch.no_grad():
        X_t = torch.tensor(scaler.transform(X), dtype=torch.float32).to(DEVICE)
        return model(X_t).cpu().numpy()

In [27]:
# 4. FEATURE ENGINEERING
def build_long_features(df_wide, ce_cols, pe_cols):
    frames = []
    for opt_type, cols_grp in [('CE', ce_cols), ('PE', pe_cols)]:
        melted = df_wide[['datetime', 'underlying_price'] + cols_grp].melt(
            id_vars=['datetime', 'underlying_price'],
            value_vars=cols_grp,
            var_name='col_name',
            value_name='iv'
        )
        melted['strike']   = melted['col_name'].map(get_strike)
        melted['opt_type'] = opt_type
        melted['is_ce']    = 1.0 if opt_type == 'CE' else 0.0
        melted['col_idx']  = melted['col_name'].map({c: i for i, c in enumerate(cols_grp)})
        frames.append(melted)

    long_df = pd.concat(frames, ignore_index=True)

    long_df['moneyness']         = long_df['strike'] / long_df['underlying_price']
    long_df['log_moneyness']     = np.log(long_df['moneyness'])
    long_df['dist_from_atm_pct'] = (
        (long_df['strike'] - long_df['underlying_price']).abs()
        / long_df['underlying_price'] * 100
    )

    # neighbor IVs via sort + groupby shift (vectorized, replaces per-row logic)
    long_df = long_df.sort_values(['datetime', 'opt_type', 'strike']).reset_index(drop=True)
    grp = long_df.groupby(['datetime', 'opt_type'])['iv']
    long_df['iv_neighbor_+1'] = grp.shift(-1)
    long_df['iv_neighbor_-1'] = grp.shift(+1)
    long_df['iv_neighbor_+2'] = grp.shift(-2)
    long_df['iv_neighbor_-2'] = grp.shift(+2)

    long_df['iv_neighbor_mean'] = long_df[
        ['iv_neighbor_-1', 'iv_neighbor_+1']
    ].mean(axis=1)

    long_df['wide_iv_neighbor_mean'] = long_df[
        ['iv_neighbor_-2', 'iv_neighbor_-1', 'iv_neighbor_+1', 'iv_neighbor_+2']
    ].mean(axis=1)

    # temporal rolling means — shift(1) prevents leakage
    long_df = long_df.sort_values(['strike', 'opt_type', 'datetime']).reset_index(drop=True)
    for window, col in [(5, 'iv_roll_mean_5'), (10, 'iv_roll_mean_10')]:
        long_df[col] = long_df.groupby(['strike', 'opt_type'])['iv'].transform(
            lambda s, w=window: s.shift(1).rolling(w, min_periods=1).mean()
        )

    # fix: for missing-iv rows, rolling mean is NaN — fill with group median
    # (filling with iv itself doesn't help when iv is also NaN)
    group_med = long_df.groupby(['strike', 'opt_type'])['iv'].transform('median')
    long_df['iv_roll_mean_5']  = long_df['iv_roll_mean_5'].fillna(group_med)
    long_df['iv_roll_mean_10'] = long_df['iv_roll_mean_10'].fillna(group_med)

    return long_df

In [28]:
# 5. MAIN
def main():
    print("Loading data...")
    df_raw           = load_raw()
    ce_cols, pe_cols = get_option_cols(df_raw)
    iv_cols          = ce_cols + pe_cols

    dates       = df_raw['datetime'].dt.date
    all_dates   = sorted(dates.unique())
    expiry_date = all_dates[-1]
    expiry_mask = (dates == expiry_date).values

    print(f"Total rows: {len(df_raw)}  |  Expiry rows: {expiry_mask.sum()}")
    print(f"Device: {DEVICE}")

    # A. Expiry day fill (cross-sectional + PCHIP)
    print("\nRunning cross-sectional + PCHIP fill for expiry day...")
    filled, _ = fill_wide(df_raw, ce_cols, pe_cols)

    # B. Build long-format features on NON-EXPIRY rows
    print("Building long-format features (non-expiry)...")
    df_non_expiry = df_raw[~expiry_mask].copy().reset_index(drop=True)
    long_df = build_long_features(df_non_expiry, ce_cols, pe_cols)

    # C. Train MLP on ALL observed non-expiry data
    print("Training MLP on all observed non-expiry data...")
    obs_all = long_df[long_df['iv'].notna()]
    X_train = obs_all[FEATURE_COLS].fillna(0).values
    y_train = obs_all['iv'].values

    # 90/10 split purely for early-stopping signal — not used for evaluation
    split = int(0.9 * len(X_train))
    model, scaler, best_ep, _, _ = train_mlp(
        X_train[:split], y_train[:split],
        X_train[split:], y_train[split:]
    )
    print(f"  Training done. best_epoch={best_ep}")

    # D. Predict missing non-expiry cells
    print("Building full long-format for prediction...")
    long_full = build_long_features(df_raw, ce_cols, pe_cols)

    long_miss = long_full[
        long_full['iv'].isna() &
        (long_full['datetime'].dt.date != expiry_date)
    ].copy()

    print(f"  MLP will fill {len(long_miss)} missing non-expiry cells.")

    if len(long_miss) > 0:
        X_miss               = long_miss[FEATURE_COLS].fillna(0).values
        long_miss['iv_pred'] = predict_mlp(model, scaler, X_miss).clip(min=0.005)
    else:
        long_miss['iv_pred'] = pd.Series(dtype=float)

    # E. Assemble final wide dataframe
    print("Assembling final wide dataframe...")
    final_wide = df_raw[['datetime', 'underlying_price'] + iv_cols].copy()
    miss_orig  = df_raw[iv_cols].isna()

    # E1. Write MLP predictions via vectorized merge (no slow for-loop)
    if len(long_miss) > 0:
        # reconstruct col_name from strike + opt_type
        long_miss['col_name'] = (
            'NIFTY27JAN26'
            + long_miss['strike'].astype(int).astype(str)
            + long_miss['opt_type']
        )

        # fix: assert lookup integrity before writing
        assert long_miss['col_name'].isin(iv_cols).all(), \
            "Some predicted col_names don't match iv_cols — check get_strike / naming"

        for col in iv_cols:
            sub = (long_miss[long_miss['col_name'] == col]
                   [['datetime', 'iv_pred']]
                   .drop_duplicates('datetime'))
            if sub.empty:
                continue
            # merge predictions aligned by datetime
            merged = final_wide[['datetime']].merge(sub, on='datetime', how='left')
            mask   = miss_orig[col].values & ~expiry_mask
            final_wide.loc[mask, col] = merged.loc[mask, 'iv_pred'].values

    # E2. Overwrite expiry missing cells with cross-sectional+PCHIP fills
    for col in iv_cols:
        mask = miss_orig[col].values & expiry_mask
        if mask.any():
            final_wide.loc[mask, col] = filled.loc[mask, col].values

    # Safety ffill/bfill for any remaining NaN
    still_missing = final_wide[iv_cols].isna().sum().sum()
    if still_missing:
        print(f"  {still_missing} cells still NaN after both fills — applying ffill/bfill")
        final_wide[iv_cols] = final_wide[iv_cols].ffill().bfill()

    final_wide[iv_cols] = final_wide[iv_cols].clip(lower=0.005)

    assert final_wide[iv_cols].isnull().sum().sum() == 0, "NaNs remain after all fills!"
    print(f"IV range: {final_wide[iv_cols].min().min():.4f} – {final_wide[iv_cols].max().max():.4f}")

    # Export filled wide CSV
    print(f"\nSaving filled wide CSV → {OUTPUT_CSV}")
    # use a copy so df_raw['datetime'] stays as Timestamp for step H
    out_wide = final_wide.copy()
    out_wide['datetime'] = df_raw['datetime'].dt.strftime('%d-%m-%Y %H:%M')
    out_wide[['datetime', 'underlying_price'] + iv_cols].to_csv(OUTPUT_CSV, index=False)

    # H. Build Kaggle submission CSV
    print(f"Building submission CSV → {SUBMIT_CSV}")
    original = pd.read_csv(RAW_CSV)
    filled   = pd.read_csv(OUTPUT_CSV)

    rows_out = []
    sub_cols = [c for c in original.columns if c not in ('datetime', 'underlying_price')]
    for col in sub_cols:
        nan_idx = original.index[original[col].isna()]
        for idx in nan_idx:
            rows_out.append({
                'id':    f"{original.loc[idx, 'datetime']}||{col}",
                'value': filled.loc[idx, col]
            })

    (pd.DataFrame(rows_out, columns=['id', 'value'])
       .sort_values('id')
       .reset_index(drop=True)
       .to_csv(SUBMIT_CSV, index=False))

    print(f"\nDone! Submission saved → {SUBMIT_CSV}")

main()

Loading data...
Total rows: 975  |  Expiry rows: 75
Device: cpu

Running cross-sectional + PCHIP fill for expiry day...
Building long-format features (non-expiry)...
Training MLP on all observed non-expiry data...
  Training done. best_epoch=191
Building full long-format for prediction...
  MLP will fill 5052 missing non-expiry cells.
Assembling final wide dataframe...
IV range: 0.0168 – 5.6968

Saving filled wide CSV → ../outputs/5_submissions/filled_dataset.csv
Building submission CSV → ../outputs/5_submissions/submission.csv

Done! Submission saved → ../outputs/5_submissions/submission.csv
